# TP 1 — Faits stylisés et efficience

**Séance 2 · 50 minutes · en binômes**


### Objectifs

Produire le **diagnostic empirique** de votre panier :

1. Tableau de statistiques descriptives comparées
2. Graphiques quantile-quantile contre la normale et contre la Student
3. Instabilité de la kurtosis et estimation de l'indice de queue
4. Corrélogrammes de $r_t$ et de $|r_t|$
5. Tests ADF et KPSS
6. Ratio de variance de Lo–MacKinlay, version robuste
7. **Note de diagnostic — 2 pages** (c'est elle qui fait la note)

### Rappel de la séance

> Le rendement de demain est imprévisible ; **son amplitude ne l'est pas.**


---

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# retrouve outils_cours.py, que le notebook soit ouvert depuis 02-TP/ ou 02-TP/corriges/
for _cand in (Path.cwd(), *list(Path.cwd().parents)[:3]):
    if (_cand / "outils_cours.py").exists():
        sys.path.insert(0, str(_cand))
        break
import outils_cours as oc

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": .3})
np.random.seed(2026)
print("Environnement prêt.")

from statsmodels.tsa.stattools import adfuller, kpss, acf

prix = oc.charger_panier()
r = oc.rendements_log(prix)
print(f"{len(r)} rendements, {prix.shape[1]} actifs.")

Environnement prêt.
⚠️  ATTENTION : données SYNTHÉTIQUES chargées (ce ne sont pas de vraies
    données de marché). Lancer 03-Data/download_data.py pour les vraies.
2099 rendements, 5 actifs.


## 1. Statistiques descriptives — et le piège du 24/7

Le panier mélange des actifs à 365 jours de cotation (cryptos) et à 252 jours
(S&P 500, or). Utiliser le même facteur d'annualisation pour tous est une erreur.

In [3]:
jours = {"Bitcoin": 365, "Ethereum": 365, "Solana": 365, "SP500": 252, "Or": 252}
table = oc.table_faits_stylises(r, jours_an=jours)
table.round(3)

,n,moy. ann. %,vol. ann. %,min %,max %,asymétrie,kurtosis excès,Jarque-Bera,p(JB),"ρ(r,1)","ρ(|r|,1)","ρ(|r|,10)"
Bitcoin,"2,099.0000",68.9680,76.2500,-21.5620,25.3020,-0.2120,3.5840,"1,138.8740",0.0000,-0.0350,0.1090,0.0490
Ethereum,"2,099.0000",108.7740,101.2060,-37.5860,40.5010,-0.0720,5.6840,"2,827.7810",0.0000,-0.0130,0.1270,0.0930
Solana,"2,099.0000",125.4620,126.5910,-44.2270,31.7440,-0.0320,3.9600,"1,371.7540",0.0000,-0.0240,0.1680,0.1100
SP500,"2,099.0000",15.7790,18.0570,-5.5450,6.7070,0.0090,2.5140,552.8190,0.0000,0.0150,0.1350,0.0550
Or,"2,099.0000",4.9490,14.2710,-5.0470,8.6370,0.5060,7.2260,"4,656.7350",0.0000,0.0270,0.0890,0.0610


**Question 1.1** — Recalculez la volatilité annualisée du Bitcoin en utilisant
252 jours au lieu de 365. De combien vous trompez-vous, en points et en pourcentage ?

In [ ]:
# À COMPLÉTER
# Comparer la volatilité annualisée du Bitcoin sous √365 et sous √252.
# Afficher : les deux valeurs, l'écart en points, la sous-estimation relative,
# et vérifier que le rapport vaut bien √(365/252).


## 2. La distribution n'est pas normale

Deux graphiques quantile-quantile : contre la normale, puis contre une Student
dont le degré de liberté est estimé sur les données.

In [ ]:
actif = "Bitcoin"          # <- changez-le pour explorer
x = r[actif].dropna()
z = (x - x.mean()) / x.std(ddof=1)

nu, _, _ = stats.t.fit(z, floc=0, fscale=1)
print(f"{actif} — degré de liberté estimé : ν = {nu:.2f}")
print(f"kurtosis en excès observée         : {stats.kurtosis(x):.2f}")
if nu > 4:
    print(f"kurtosis théorique d'une t(ν)      : {6/(nu-4):.2f}")
else:
    print("ν ≤ 4 → la kurtosis théorique de la Student est INFINIE.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
stats.probplot(z, dist="norm", plot=axes[0])
axes[0].set_title(f"{actif} — QQ contre la normale")
stats.probplot(z, dist=stats.t, sparams=(nu,), plot=axes[1])
axes[1].set_title(f"{actif} — QQ contre une Student(ν={nu:.1f})")
for a in axes:
    a.get_lines()[0].set(marker=".", markersize=3, alpha=.6)
plt.tight_layout(); plt.show()

**Question 2.1** — Sur le graphique de gauche, où les points s'écartent-ils
de la droite ? Que signifie physiquement cet écart pour un gérant de risque ?

**Question 2.2** — La Student ajuste-t-elle mieux ? Sur quelle partie de la
distribution reste-t-elle insuffisante, et pourquoi ?

*Vos réponses :*

>

## 3. La kurtosis se stabilise-t-elle ?

Si le moment d'ordre 4 existe, la kurtosis empirique converge quand l'échantillon
grandit. Si elle **continue de croître**, c'est que le moment n'existe pas.

In [ ]:
# À COMPLÉTER
# Tracer la kurtosis en excès calculée sur les n premières observations,
# pour n allant de 250 à la taille totale par pas de 100.
# Conclure : se stabilise-t-elle ?


**Question 3.1** — Estimez l'indice de queue par la méthode de Hill, pour
plusieurs valeurs de $k$. Y a-t-il une zone de stabilité ? Que vaut $\alpha$ ?
Le moment d'ordre 4 existe-t-il ?

In [ ]:
# À COMPLÉTER
# Tracer alpha_hat de Hill en fonction de k (utiliser oc.hill).
# Ajouter les lignes de repère alpha = 2 et alpha = 4.
# Conclure sur l'existence du moment d'ordre 4.


## 4. Dépendance : rendements vs amplitudes

C'est le fait stylisé qui rend possible toute la séance 3.

In [ ]:
n_lags = 40
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, serie, titre in [
    (axes[0], x, "rendements  $r_t$"),
    (axes[1], x.abs(), "amplitudes  $|r_t|$"),
]:
    vals = acf(serie, nlags=n_lags, fft=True)[1:]
    ax.bar(range(1, n_lags + 1), vals, width=.7)
    ic = 1.96 / np.sqrt(len(serie))
    ax.axhline(ic, color="crimson", ls="--", lw=.9)
    ax.axhline(-ic, color="crimson", ls="--", lw=.9)
    ax.set_title(f"{actif} — autocorrélation des {titre}")
    ax.set_xlabel("retard (jours)")
    ax.set_ylim(-0.15, max(0.4, vals.max() * 1.2))
plt.tight_layout(); plt.show()

**Question 4.1** — Combien de retards sont significatifs pour $r_t$ ?
Pour $|r_t|$ ? Formulez la conclusion en une phrase.

*Votre réponse :*

>

**Question 4.2 (bonus)** — Testez l'effet de levier : corrélation entre le
rendement du jour $t$ et la volatilité réalisée des 5 jours suivants.
Comparez Bitcoin et S&P 500. Le résultat est-il celui attendu ?

In [ ]:
# À COMPLÉTER
# Pour Bitcoin puis SP500 : corrélation entre r_t et l'écart-type des
# rendements sur [t+1, t+5]. Indice : .rolling(5).std().shift(-5)


## 5. Racine unitaire : ADF et KPSS

Rappel : ADF a pour $H_0$ « racine unitaire », KPSS a pour $H_0$ « stationnarité ».
Les deux sont complémentaires.

In [ ]:
def tests_racine(serie, nom):
    adf_p = adfuller(serie.dropna(), autolag="AIC")[1]
    kpss_p = kpss(serie.dropna(), regression="c", nlags="auto")[1]
    return {
        "série": nom,
        "p(ADF)": round(adf_p, 4),
        "ADF rejette H0 ?": "oui" if adf_p < 0.05 else "non",
        "p(KPSS)": round(kpss_p, 4),
        "KPSS rejette H0 ?": "oui" if kpss_p < 0.05 else "non",
        "conclusion": (
            "stationnaire" if adf_p < 0.05 and kpss_p >= 0.05
            else "NON stationnaire" if adf_p >= 0.05 and kpss_p < 0.05
            else "ambigu — mémoire longue ou rupture ?"
        ),
    }

resultats = []
for c in prix.columns:
    resultats.append(tests_racine(np.log(prix[c]), f"log-prix {c}"))
for c in r.columns:
    resultats.append(tests_racine(r[c], f"rendements {c}"))
pd.DataFrame(resultats).set_index("série")

**Question 5.1** — Le résultat sur les log-prix est-il surprenant ?
Vaut-il spécifiquement pour les crypto-actifs ?

**Question 5.2** — Un camarade écrit : *« ADF ne rejette pas sur le Bitcoin,
donc le Bitcoin suit une marche aléatoire et le marché est efficient. »*
Relevez les **deux** erreurs de raisonnement.

*Vos réponses :*

>

## 6. Ratio de variance de Lo–MacKinlay

Le vrai test de la marche aléatoire. On utilise **impérativement** la version
robuste à l'hétéroscédasticité — au vu de la partie 4, la version standard
rejetterait pour la mauvaise raison.

In [ ]:
lignes = []
for c in r.columns:
    for q in (2, 4, 8, 16):
        res = oc.variance_ratio(r[c], q, robuste=True)
        lignes.append({
            "actif": c, "q": q,
            "VR": round(res["VR"], 3),
            "z": round(res["z"], 2),
            "p": round(res["p_value"], 4),
            "rejet à 5 %": "oui" if res["p_value"] < 0.05 else "non",
        })
vr_table = pd.DataFrame(lignes).pivot(index="actif", columns="q",
                                      values=["VR", "p"]).round(3)
vr_table

**Question 6.1** — Pour quels actifs et quels horizons la marche aléatoire
est-elle rejetée ? Le rejet est-il concentré à horizon court ou long ?

**Question 6.2** — Comparez avec la version **non robuste**. Que se passe-t-il ?
Expliquez pourquoi.

In [ ]:
# À COMPLÉTER
# Comparer, pour q=4 et pour chaque actif, la p-value robuste et la p-value
# non robuste (argument robuste=False). Que constatez-vous, et pourquoi ?


**Question 6.3 (extension possible en projet)** — L'efficience du Bitcoin
a-t-elle changé après le lancement des ETF au comptant (janvier 2024) ?
Comparez $VR(q)$ avant et après.

In [ ]:
# À COMPLÉTER (facultatif en séance, obligatoire si vous prenez le sujet 1)
# Découper la série BTC avant / après le 2024-01-11 et comparer VR(q).
# N'oubliez pas de commenter la PUISSANCE du test sur le sous-échantillon court.


---

## 7. Note de diagnostic — à rendre

**2 pages maximum**, en binôme, accompagnée de ce notebook exécuté.

Structure attendue :

1. **Données et choix méthodologiques** (½ page) — période, actifs, calendrier
   retenu et **justification** du facteur d'annualisation.
2. **Faits stylisés** (1 page) — le tableau, le QQ-plot, les corrélogrammes, avec
   une phrase d'interprétation par résultat. Comparaison crypto / traditionnel.
3. **Efficience** (½ page) — résultats ADF/KPSS/VR et **conclusion argumentée**.

### Ce qui est évalué

| | |
|---|---|
| Exactitude technique | 30 % |
| **Qualité de l'interprétation** | **40 %** |
| Rigueur méthodologique (justification des choix) | 20 % |
| Reproductibilité du notebook | 10 % |

> **Rappel**
>
> « Le test ADF donne p = 0,43 » n'est pas une interprétation, c'est une lecture.
> On attend : ce que ce résultat implique, et ce qu'il n'implique pas.
